In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import chromadb

/home/shubham/build_in_public/pdf-chatbot/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
model_name = "meta-llama/Llama-3.1-8B-Instruct"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map=device)
print(model)

Loading checkpoint shards: 100%|██████████| 4/4 [00:08<00:00,  2.03s/it]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
    (rotary_

In [34]:
sequence = ["I've been waiting for a HuggingFace course my whole life.", 
            "Can't wait to finish this course"]
tokenizer.pad_token_id = 0 
output = tokenizer(sequence, return_tensors="pt", padding="max_length", 
        truncation=True, max_length=10)


In [ ]:
output['input_ids'].shape

In [ ]:
output['input_ids'][0]

In [ ]:
output['input_ids']

In [ ]:
for input_i in output['input_ids']:
    print(input_i)
    print(tokenizer.convert_ids_to_tokens(input_i))

In [ ]:
# tokenizer
tokenizer.pad_token_id = tokenizer.eos_token_id
print(tokenizer.pad_token_id)

In [4]:
# getting docs based on user query 
def get_chrome_client(persist_directory, collection_name):
    # Initialize ChromaDB client with updated configuration
    chroma_client = chromadb.PersistentClient(path=persist_directory)
    print(f"ChromaDB client initialized with persistence at: {persist_directory}")
   
    try:
        collection = chroma_client.get_collection(name=collection_name)
        print(f"Using existing collection: {collection_name}")
    except:
        collection = chroma_client.create_collection(
            name=collection_name,
            metadata={"description": "Government of India Budget 2025-2026 documents"}
        )
        print(f"Created new collection: {collection_name}")
    return chroma_client, collection

In [5]:
collection_name = "budget_rag"
persist_directory = "/home/shubham/build_in_public/pdf-chatbot/chroma_db"
chroma_client, collection = get_chrome_client(persist_directory, collection_name)

ChromaDB client initialized with persistence at: /home/shubham/build_in_public/pdf-chatbot/chroma_db
Using existing collection: budget_rag


In [6]:
user_query = "what is the credit card limit for micro enterprises?"
retrieved_docs = collection.query(query_texts=[user_query], n_results=3)
print(len(retrieved_docs))

8


In [7]:
len(retrieved_docs['documents'][0]), retrieved_docs['documents'][0]

(3,
 ['**Credit Cards for Micro Enterprises**  \n**30.** We will introduce customized Credit Cards with a ` 5 lakh limit for micro',
  '**29.** To improve access to credit, the credit guarantee cover will be\nenhanced:  \na) For Micro and Small Enterprises, from ` 5 crore to 10 crore, leading',
  'linked credit cards with ` 30,000 limit, and capacity building support.  \n**Social Security Scheme for Welfare of Online Platform Workers**'])

In [8]:
retrieved_docs = "\n".join(retrieved_docs.get("documents", [])[0])

In [9]:
prompt_with_rag = f"""You are an AI assistant that answers questions based on provided documents.
    
Context:
{retrieved_docs}

Question: {user_query}
Answer:
"""

In [ ]:
# padding off for single user prompt 

encoded_input = tokenizer(prompt_with_rag, return_tensors='pt', 
                   truncation=True, max_length=512).to(device)

encoded_input

In [ ]:
with torch.no_grad():
    output = model.generate(**encoded_input, max_new_tokens=60)

In [ ]:
print(tokenizer.decode(output[0], skip_special_tokens=True))

In [89]:
another_prompt = f"""<|begin_of_text|>
<|start_of_turn|><|system|>
You are a helpful assistant that answers questions based on provided documents.
<|end_of_turn|>
<|start_of_turn|><|user|>
{user_query}
Context:
{retrieved_docs}
<|end_of_turn|>
<|start_of_turn|><|assistant|>
"""

In [ ]:
# padding off for single user prompt 

encoded_input = tokenizer(prompt_with_rag, return_tensors='pt', 
                   truncation=True, max_length=512).to(device)
with torch.no_grad():
    output = model.generate(**encoded_input, max_new_tokens=60)
print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
output.shape

`another_prompt`: This prompting style is better as it follows `Llama-3.1-8B-Instruct` 

In [12]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that answers questions based on provided documents."},
    {"role": "user", "content": f"Context:\n{retrieved_docs}\n\nQuestion: {user_query}"},
]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(formatted_prompt)

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant that answers questions based on provided documents.<|eot_id|><|start_header_id|>user<|end_header_id|>

Context:
**Credit Cards for Micro Enterprises**  
**30.** We will introduce customized Credit Cards with a ` 5 lakh limit for micro
**29.** To improve access to credit, the credit guarantee cover will be
enhanced:  
a) For Micro and Small Enterprises, from ` 5 crore to 10 crore, leading
linked credit cards with ` 30,000 limit, and capacity building support.  
**Social Security Scheme for Welfare of Online Platform Workers**

Question: what is the credit card limit for micro enterprises?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [18]:
encoded_input = tokenizer(prompt_with_rag, return_tensors='pt', 
                   truncation=True, max_length=512).to(device)
with torch.no_grad():
    output = model.generate(**encoded_input, max_new_tokens=60)

print(output.shape)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


torch.Size([1, 187])


In [20]:
print(tokenizer.decode(output[0], skip_special_tokens=True))

You are an AI assistant that answers questions based on provided documents.

Context:
**Credit Cards for Micro Enterprises**  
**30.** We will introduce customized Credit Cards with a ` 5 lakh limit for micro
**29.** To improve access to credit, the credit guarantee cover will be
enhanced:  
a) For Micro and Small Enterprises, from ` 5 crore to 10 crore, leading
linked credit cards with ` 30,000 limit, and capacity building support.  
**Social Security Scheme for Welfare of Online Platform Workers**

Question: what is the credit card limit for micro enterprises?
Answer:
The credit card limit for micro enterprises is ` 5 lakh.  Additionally, linked credit cards with a ` 30,000 limit will also be available.  However, the main limit mentioned is ` 5 lakh.  Therefore, the answer is ` 5 lakh.  The linked credit
